# Работу выполнил студент ДПИ25-1 Карчевский Андрей

# Pandas — Лабораторная работа №2

## Разминка

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("")


### 1) Загрузка `sp500hst.txt` и назначение столбцов

In [2]:
sp_path = DATA_DIR / "sp500hst.txt"

cols = ["date", "ticker", "open", "high", "low", "close", "volume"]
sp = pd.read_csv(sp_path, header=None, names=cols)

# Приведём дату к типу datetime для дальнейших операций
sp["date"] = pd.to_datetime(sp["date"].astype(str), format="%Y%m%d")

sp.head()

,date,ticker,open,high,low,close,volume
0,2009-08-21,A,25.60,25.6100,25.220,25.55,34758
1,2009-08-24,A,25.64,25.7400,25.330,25.50,22247
2,2009-08-25,A,25.50,25.7000,25.225,25.34,30891
3,2009-08-26,A,25.32,25.6425,25.145,25.48,33334
4,2009-08-27,A,25.50,25.5700,25.230,25.54,70176


### 2) Среднее по столбцам 3–6 (open, high, low, close)

In [3]:
means_3_6 = sp[["open", "high", "low", "close"]].mean()
means_3_6

open     42.611619
high     43.118704
low      42.070369
close    42.618050
dtype: float64

### 3) Добавление столбца с номером месяца

In [4]:
sp["month"] = sp["date"].dt.month
sp[["date", "month"]].head()

,date,month
0,2009-08-21,8
1,2009-08-24,8
2,2009-08-25,8
3,2009-08-26,8
4,2009-08-27,8


### 4) Суммарный объём торгов для каждого тикера

In [5]:
volume_by_ticker = sp.groupby("ticker", as_index=False)["volume"].sum().sort_values("volume", ascending=False)
volume_by_ticker.head(10)

,ticker,volume
74,C,1458332551
49,BAC,465813622
176,F,230212585
200,GE,204452485
241,INTC,159979834
321,MSFT,148263502
414,S,146922025
372,PFE,141891968
169,ETFC,122969972
117,CSCO,121933642


### 5) Добавление расшифровки тикера из `sp_data2.csv` (корректная обработка пропусков)

In [6]:
sp_names_path = DATA_DIR / "sp_data2.csv"

# Файл разделён точкой с запятой и не содержит заголовка
sp_names = pd.read_csv(sp_names_path, sep=";", header=None, names=["ticker", "ticker_name", "weight"])

sp_enriched = sp.merge(sp_names[["ticker", "ticker_name"]], on="ticker", how="left")

# Если для тикера нет расшифровки — оставим сам тикер как запасной вариант
sp_enriched["ticker_name"] = sp_enriched["ticker_name"].fillna(sp_enriched["ticker"])

sp_enriched[["ticker", "ticker_name"]].drop_duplicates().head(10)

,ticker,ticker_name
0,A,Agilent Technologies
245,AA,AA
490,AAPL,Apple
731,ABC,AmerisourceBergen
826,ABT,Abbott Laboratories
1071,ACE,ACE
1128,ACS,ACS
1238,ADBE,Adobe Systems
1481,ADI,Analog Devices
1726,ADM,Archer-Daniels Midland


## Лабораторная работа №2

Далее используются датасеты `recipes_sample.csv` и `reviews_sample.csv`.

### 1) Базовые операции с `DataFrame`

#### 1.1 Загрузка `recipes` и `reviews` (учёт безымянного столбца-индекса)

In [7]:
recipes_path = DATA_DIR / "recipes_sample.csv"
reviews_path = DATA_DIR / "reviews_sample.csv"

# В recipes просто читаем таблицу
recipes = pd.read_csv(recipes_path)

# В reviews первый столбец без имени — это индекс, поэтому читаем его как index_col=0
reviews = pd.read_csv(reviews_path, index_col=0)

recipes.head(), reviews.head()

(                                       name     id  minutes  contributor_id  \
 0     george s at the cove  black bean soup  44123       90           35193   
 1        healthy for them  yogurt popsicles  67664       10           91970   
 2              i can t believe it s spinach  38798       30            1533   
 3                      italian  gut busters  35173       45           22724   
 4  love is in the air  beef fondue   sauces  84797       25            4470   
 
     submitted  n_steps                                        description  \
 0  2002-10-25      NaN  an original recipe created by chef scott meska...   
 1  2003-07-26      NaN  my children and their friends ask for my homem...   
 2  2002-08-29      NaN            these were so go, it surprised even me.   
 3  2002-07-27      NaN  my sister-in-law made these for us at a family...   
 4  2004-02-23      4.0  i think a fondue is a very romantic casual din...   
 
    n_ingredients  
 0           18.0  
 1      

#### 1.2 Основные параметры таблиц (размеры и типы столбцов)

In [8]:
def describe_df(df: pd.DataFrame, name: str) -> None:
    print(f"Таблица {name}:")
    print(f"  строк: {df.shape[0]}")
    print(f"  столбцов: {df.shape[1]}")
    print("  типы столбцов:")
    print(df.dtypes)
    print()

describe_df(recipes, "recipes")
describe_df(reviews, "reviews")

Таблица recipes:
  строк: 30000
  столбцов: 8
  типы столбцов:
name                  str
id                  int64
minutes             int64
contributor_id      int64
submitted             str
n_steps           float64
description           str
n_ingredients     float64
dtype: object

Таблица reviews:
  строк: 126696
  столбцов: 5
  типы столбцов:
user_id      int64
recipe_id    int64
date           str
rating       int64
review         str
dtype: object



#### 1.3 Пропуски: в каких столбцах есть NaN и доля строк с пропусками

In [9]:
def missing_info(df: pd.DataFrame) -> pd.DataFrame:
    cols_with_na = df.isna().any()
    na_counts = df.isna().sum()
    out = pd.DataFrame({
        "has_na": cols_with_na,
        "na_count": na_counts,
        "na_share": (na_counts / len(df)).round(6),
    })
    return out[out["has_na"]].sort_values("na_count", ascending=False)

recipes_missing = missing_info(recipes)
reviews_missing = missing_info(reviews)

recipes_rows_with_na_share = df_rows_na_share = recipes.isna().any(axis=1).mean()
reviews_rows_with_na_share = reviews.isna().any(axis=1).mean()

recipes_missing, recipes_rows_with_na_share, reviews_missing, reviews_rows_with_na_share

(               has_na  na_count  na_share
 n_steps          True     11190  0.373000
 n_ingredients    True      8880  0.296000
 description      True       623  0.020767,
 np.float64(0.5684666666666667),
         has_na  na_count  na_share
 review    True        17  0.000134,
 np.float64(0.00013417945317926376))

#### 1.4 Средние значения по числовым столбцам

In [10]:
recipes_numeric_means = recipes.select_dtypes(include=[np.number]).mean(numeric_only=True)
reviews_numeric_means = reviews.select_dtypes(include=[np.number]).mean(numeric_only=True)

recipes_numeric_means, reviews_numeric_means

(id                2.218793e+05
 minutes           1.233581e+02
 contributor_id    5.635901e+06
 n_steps           9.805582e+00
 n_ingredients     9.008286e+00
 dtype: float64,
 user_id      1.408013e+08
 recipe_id    1.600944e+05
 rating       4.410802e+00
 dtype: float64)

#### 1.5 Серия из 10 случайных названий рецептов

In [11]:
random_recipe_names = recipes["name"].dropna().sample(10, random_state=42).reset_index(drop=True)
random_recipe_names

0                                   barbecued potatoes
1                                reena s pickled beets
2        saucy mocha pots of cream  microwave easy fix
3                                      spaghetti salad
4                     beef steaks with capsicum relish
5                crispy low carb fried chicken nuggets
6    cheese ravioli with white wine sauce and zucchini
7                            cowboy sourdough pancakes
8                                 french toast muffins
9                              old fashion egg custard
Name: name, dtype: str

#### 1.6 Переиндексация `reviews` с нуля

In [12]:
reviews_reset = reviews.reset_index(drop=True)
reviews_reset.head()

,user_id,recipe_id,date,rating,review
0,21752,57993,2003-05-01,5,Last week whole sides of frozen salmon fillet ...
1,431813,142201,2007-09-16,5,So simple and so tasty! I used a yellow capsi...
2,400708,252013,2008-01-10,4,"Very nice breakfast HH, easy to make and yummy..."
3,2001852463,404716,2017-12-11,5,These are a favorite for the holidays and so e...
4,95810,129396,2008-03-14,5,Excellent soup! The tomato flavor is just gre...


#### 1.7 Рецепты: время ≤ 20 минут и ингредиентов ≤ 5

In [13]:
quick_and_simple = recipes[
    (recipes["minutes"] <= 20) &
    (recipes["n_ingredients"] <= 5)
][["id", "name", "minutes", "n_ingredients"]]

quick_and_simple.head(20)

,id,name,minutes,n_ingredients
28,302399,quick biscuit bread,20,5.0
60,303944,peas fit for a king or queen,20,5.0
90,100837,hawaiian sunrise mimosa,5,3.0
91,286484,tasty dish s banana pudding in 2 minutes,2,4.0
94,11361,1 minute meatballs,13,2.0
112,513963,10 minute lime pie,10,5.0
117,126072,100 year old pie crust,15,5.0
121,121712,1000 island vegetable dip,20,2.0
143,256464,2 minute broccoli,4,2.0
165,112648,3 fruit salad,15,5.0


### 2) Работа с датами в `pandas`

#### 2.1 Преобразование `submitted` в datetime и чтение сразу в нужном формате

In [14]:
# Перечитываем recipes, сразу парся дату
recipes = pd.read_csv(recipes_path, parse_dates=["submitted"])

recipes.dtypes

name                         str
id                         int64
minutes                    int64
contributor_id             int64
submitted         datetime64[us]
n_steps                  float64
description                  str
n_ingredients            float64
dtype: object

#### 2.2 Рецепты, добавленные не позже 2010 года

In [15]:
recipes_upto_2010 = recipes[recipes["submitted"] <= "2010-12-31"][["id", "name", "submitted"]]
recipes_upto_2010.head(10), len(recipes_upto_2010)

(        id                                      name  submitted
 0    44123     george s at the cove  black bean soup 2002-10-25
 1    67664        healthy for them  yogurt popsicles 2003-07-26
 2    38798              i can t believe it s spinach 2002-08-29
 3    35173                      italian  gut busters 2002-07-27
 4    84797  love is in the air  beef fondue   sauces 2004-02-23
 5    44045                  mennonite  corn fritters 2002-10-25
 6   107229                      open sesame  noodles 2004-12-30
 7    95926                say what   banana sandwich 2004-07-20
 9   306168                    412 broccoli casserole 2008-05-30
 10   50662                          25 000 casserole 2003-01-10,
 27661)

### 3) Работа со строковыми данными в `pandas`

#### 3.1 `description_length` — длина описания

In [16]:
recipes["description_length"] = recipes["description"].fillna("").str.len()
recipes[["id", "description_length"]].head()

,id,description_length
0,44123,330
1,67664,255
2,38798,39
3,35173,154
4,84797,587


#### 3.2 Название рецепта: каждое слово с прописной буквы

In [17]:
recipes["name"] = recipes["name"].astype(str).str.title()
recipes[["id", "name"]].head()

,id,name
0,44123,George S At The Cove Black Bean Soup
1,67664,Healthy For Them Yogurt Popsicles
2,38798,I Can T Believe It S Spinach
3,35173,Italian Gut Busters
4,84797,Love Is In The Air Beef Fondue Sauces


#### 3.3 `name_word_count` — количество слов в названии (учёт нескольких пробелов)

In [18]:
recipes["name_word_count"] = recipes["name"].astype(str).str.split().str.len()
recipes[["id", "name", "name_word_count"]].head()

,id,name,name_word_count
0,44123,George S At The Cove Black Bean Soup,8
1,67664,Healthy For Them Yogurt Popsicles,5
2,38798,I Can T Believe It S Spinach,7
3,35173,Italian Gut Busters,3
4,84797,Love Is In The Air Beef Fondue Sauces,8


### 4) Группировки таблиц `pd.DataFrame`

#### 4.1 Кол-во рецептов по `contributor_id` и максимум

In [19]:
recipes_by_contributor = recipes["contributor_id"].value_counts()
top_contributor_id = recipes_by_contributor.idxmax()
top_contributor_count = int(recipes_by_contributor.max())

top_contributor_id, top_contributor_count

(np.int64(89831), 421)

#### 4.2 Средний рейтинг по рецептам и сколько рецептов без отзывов

In [20]:
# Средний рейтинг по каждому рецепту (по имеющимся отзывам)
avg_rating_by_recipe = reviews.groupby("recipe_id")["rating"].mean()

# Рецепты без отзывов — те, id которых отсутствуют в reviews.recipe_id
recipe_ids = set(recipes["id"].unique())
reviewed_recipe_ids = set(reviews["recipe_id"].unique())
no_review_ids = recipe_ids - reviewed_recipe_ids

len(no_review_ids), avg_rating_by_recipe.head()

(1900,
 recipe_id
 48    1.000000
 55    4.750000
 66    4.944444
 91    4.750000
 94    5.000000
 Name: rating, dtype: float64)

#### 4.3 Количество рецептов по годам добавления

In [21]:
recipes_by_year = recipes["submitted"].dt.year.value_counts().sort_index()
recipes_by_year.head(10)

submitted
1999     275
2000     104
2001     589
2002    2644
2003    2334
2004    2153
2005    3130
2006    3473
2007    4429
2008    4029
Name: count, dtype: int64

### 5) Объединение таблиц `pd.DataFrame`

#### 5.1 Таблица `id`, `name`, `user_id`, `rating` только для рецептов с отзывами

In [22]:
recipes_min = recipes[["id", "name", "submitted"]]
reviews_min = reviews[["user_id", "recipe_id", "rating"]]

recipes_with_ratings = recipes_min.merge(
    reviews_min,
    left_on="id",
    right_on="recipe_id",
    how="inner"
)[["id", "name", "user_id", "rating"]]

# Проверка: выберем рецепт без отзывов и убедимся, что его нет в результате
no_review_example_id = next(iter(no_review_ids))
is_present = (recipes_with_ratings["id"] == no_review_example_id).any()

no_review_example_id, is_present, recipes_with_ratings.head()

(np.int64(401411),
 np.False_,
       id                                   name  user_id  rating
 0  44123  George S At The Cove  Black Bean Soup   743566       5
 1  44123  George S At The Cove  Black Bean Soup    76503       5
 2  44123  George S At The Cove  Black Bean Soup    34206       5
 3  67664     Healthy For Them  Yogurt Popsicles   494084       5
 4  67664     Healthy For Them  Yogurt Popsicles   303445       5)

#### 5.2 Таблица `recipe_id`, `name`, `review_count` (для отсутствующих отзывов — 0)

In [28]:
review_counts = reviews.groupby("recipe_id").size().rename("review_count")

recipes_with_review_count = recipes[["id", "name"]].merge(
    review_counts,
    left_on="id",
    right_index=True,
    how="left"
).rename(columns={"id": "recipe_id"})

recipes_with_review_count["review_count"] = recipes_with_review_count["review_count"].fillna(0).astype(int)

# Проверка для рецепта без отзывов
recipes_with_review_count.loc[recipes_with_review_count["recipe_id"] == no_review_example_id].head()

,recipe_id,name,review_count
8828,401411,Crunchy Ranch Croutons,0


#### 5.3 В каком году добавленные рецепты имеют наименьший средний рейтинг?

In [29]:
# Присоединим год добавления рецепта к каждой оценке и посчитаем средний рейтинг по годам
ratings_with_year = recipes[["id", "submitted"]].merge(
    reviews[["recipe_id", "rating"]],
    left_on="id",
    right_on="recipe_id",
    how="inner"
)
ratings_with_year["year"] = ratings_with_year["submitted"].dt.year

mean_rating_by_year = ratings_with_year.groupby("year")["rating"].mean().sort_values()
mean_rating_by_year.head(10)

year
2017    2.750000
2018    3.388889
2016    3.538462
2015    4.207317
1999    4.274895
2000    4.284585
2013    4.336508
2009    4.360447
2011    4.375850
2008    4.387416
Name: rating, dtype: float64

### 6) Сохранение таблиц `pd.DataFrame`

#### 6.1 Сортировка по `name_word_count` и сохранение результатов 3.1–3.3 в CSV

In [30]:
recipes_sorted = recipes.sort_values("name_word_count", ascending=False)

out_csv = DATA_DIR / "recipes_text_features.csv"
recipes_sorted.to_csv(out_csv, index=False)

out_csv, recipes_sorted[["id", "name", "description_length", "name_word_count"]].head()

(PosixPath('recipes_text_features.csv'),
            id                                               name  \
 26223   77188  Subru Uncle S Whole Green Moong Dal I Ll Be Ma...   
 28083  102274  Tsr Version Of T G I  Friday S Black Bean Soup...   
 26222   76908  Subru Uncle S Toor Ki Dal Sindhi Style  Dad  M...   
 27876  113346  Top Secret Recipes Version Of  I H O P  Griddl...   
 5734   294898  Chicken Curry Or  Cat S Vomit On A Bed Of Magg...   
 
        description_length  name_word_count  
 26223                 343               15  
 28083                 436               14  
 26222                1087               14  
 27876                 129               14  
 5734                  144               13  )

#### 6.2 Сохранение результатов 5.1 и 5.2 в Excel (разные листы)

In [32]:
out_xlsx = DATA_DIR / "recipes_reviews.xlsx"

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    recipes_with_ratings.to_excel(writer, sheet_name="Рецепты с оценками", index=False)
    recipes_with_review_count.to_excel(writer, sheet_name="Количество отзывов по рецептам", index=False)

out_xlsx

PosixPath('recipes_reviews.xlsx')